# Prévision de pluie en Australie — 3/3 · Modélisation & évaluation

**Cette partie** : Régression Logistique, Random Forest (+ variante `balanced`), comparatif vs baselines, conclusions & pistes MLOps.

> **Notebook indépendant** — il recharge les données depuis zéro (chaque partie s'exécute seule). Voir aussi : `01_exploration.ipynb`, `02_preprocessing.ipynb`, `03_modelisation.ipynb`.

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)
RANDOM_STATE = 42

In [ ]:
# Résolution robuste du chemin des données : fonctionne que le notebook soit
# lancé depuis la racine du projet ou depuis le dossier notebooks/.
CANDIDATES = [
    Path("Data/weatherAUS.csv"),
    Path("../Data/weatherAUS.csv"),
    Path.home() / "Desktop/MlOps_Meteo-Liora/Data/weatherAUS.csv",
]
DATA_PATH = next((p for p in CANDIDATES if p.exists()), None)
assert DATA_PATH is not None, "weatherAUS.csv introuvable (vérifier le dossier Data/)"

df = pd.read_csv(DATA_PATH, na_values=["NA"])
print(f"Chargé depuis : {DATA_PATH}")
print(f"Dimensions    : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
df.head()

### Rappel — préprocessing
On reconstruit ici le pipeline de préparation (détaillé dans `02_preprocessing.ipynb`) pour que ce notebook soit exécutable seul.

In [ ]:
data = df.drop(columns=["_month"], errors="ignore").copy()
data = data.dropna(subset=["RainTomorrow"]).copy()

# Cible 0/1
y = (data["RainTomorrow"] == "Yes").astype(int)

# Feature temporelle
data["Month"] = pd.to_datetime(data["Date"], errors="coerce").dt.month.astype("Int64")
data = data.drop(columns=["Date", "RainTomorrow"])

# Listes de features
categorical_features = ["Location", "WindGustDir", "WindDir9am", "WindDir3pm", "RainToday", "Month"]
numeric_features = [c for c in data.columns if c not in categorical_features]
X = data

print(f"Échantillons : {len(X):,}  |  numériques : {len(numeric_features)}  |  catégorielles : {len(categorical_features)}")
print("Numériques  :", numeric_features)
print("Catégoriques:", categorical_features)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features),
])
preprocessor

## 3. Modélisation

Deux modèles classiques pour la classification binaire — **Régression Logistique** (référence linéaire)
et **Random Forest** (ensemble non linéaire) — chacun encapsulé dans un pipeline avec le préprocesseur,
donc **ajusté uniquement sur le train**. Métriques : accuracy, *recall*/F1 de la classe « pluie », ROC-AUC.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_auc_score, recall_score, f1_score)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
print(f"Train : {X_train.shape[0]:,}  |  Test : {X_test.shape[0]:,}")

### 3.1 Régression Logistique

In [ ]:
pipe_lr = Pipeline([("prep", preprocessor),
                    ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))])
pipe_lr.fit(X_train, y_train)
y_pred_lr = pipe_lr.predict(X_test)
proba_lr = pipe_lr.predict_proba(X_test)[:, 1]

print(f"Accuracy : {accuracy_score(y_test, y_pred_lr):.4f}   ROC-AUC : {roc_auc_score(y_test, proba_lr):.4f}")
print(classification_report(y_test, y_pred_lr, target_names=["No", "Yes"]))

cm = confusion_matrix(y_test, y_pred_lr)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["No", "Yes"], yticklabels=["No", "Yes"])
plt.title("Matrice de confusion — Régression Logistique")
plt.xlabel("Prédit"); plt.ylabel("Réel"); plt.tight_layout(); plt.show()

**Lecture.** Bonne accuracy globale (~0,85), mais le *recall* sur la classe « pluie » reste
modeste (~0,5) : le modèle rate environ la moitié des jours de pluie — conséquence directe du
déséquilibre. La ROC-AUC (~0,87) montre tout de même un bon pouvoir discriminant.

### 3.2 Random Forest

In [ ]:
pipe_rf = Pipeline([("prep", preprocessor),
                    ("clf", RandomForestClassifier(n_estimators=100, n_jobs=-1,
                                                   random_state=RANDOM_STATE))])
pipe_rf.fit(X_train, y_train)
y_pred_rf = pipe_rf.predict(X_test)
proba_rf = pipe_rf.predict_proba(X_test)[:, 1]

print(f"Accuracy : {accuracy_score(y_test, y_pred_rf):.4f}   ROC-AUC : {roc_auc_score(y_test, proba_rf):.4f}")
print(classification_report(y_test, y_pred_rf, target_names=["No", "Yes"]))

cm = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["No", "Yes"], yticklabels=["No", "Yes"])
plt.title("Matrice de confusion — Random Forest")
plt.xlabel("Prédit"); plt.ylabel("Réel"); plt.tight_layout(); plt.show()

In [ ]:
# Importance des variables (top 15)
feat_names = pipe_rf.named_steps["prep"].get_feature_names_out()
importances = pd.Series(pipe_rf.named_steps["clf"].feature_importances_, index=feat_names)
top15 = importances.sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top15.index[::-1], top15.values[::-1], color="#8172B3")
ax.set_title("Random Forest — 15 variables les plus importantes")
plt.tight_layout(); plt.show()
top15.round(4)

**Lecture.** Le Random Forest fait légèrement mieux (accuracy ~0,855, ROC-AUC ~0,88) mais
souffre du **même recall faible** sur la classe « pluie ». Les variables dominantes confirment l'EDA :
`Humidity3pm`, `Sunshine`, `Pressure`, `Rainfall`, `WindGustSpeed`.

### 3.3 Comparatif — modèles vs baselines

Bonus : une régression logistique `class_weight="balanced"` pour illustrer l'effet de la prise en
compte du déséquilibre (recall « pluie » ↑ au prix de l'accuracy globale ↓).

In [ ]:
pipe_lr_bal = Pipeline([("prep", preprocessor),
                        ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE,
                                                   class_weight="balanced"))])
pipe_lr_bal.fit(X_train, y_train)
y_pred_bal = pipe_lr_bal.predict(X_test)
proba_bal = pipe_lr_bal.predict_proba(X_test)[:, 1]

def row(name, y_pred, proba=None):
    return {
        "Modèle": name,
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "Recall (pluie)": round(recall_score(y_test, y_pred), 4),
        "F1 (pluie)": round(f1_score(y_test, y_pred), 4),
        "ROC-AUC": round(roc_auc_score(y_test, proba), 4) if proba is not None else np.nan,
    }

# baseline "toujours Non" sur le test
y_pred_zero = np.zeros_like(y_test)

compare = pd.DataFrame([
    row("Baseline toujours-Non", y_pred_zero),
    row("Régression Logistique", y_pred_lr, proba_lr),
    row("Random Forest", y_pred_rf, proba_rf),
    row("Rég. Log. (balanced)", y_pred_bal, proba_bal),
]).set_index("Modèle")
compare

## 4. Conclusions & pistes MLOps

**Synthèse data.**
- Dataset propre dans l'ensemble (0 doublon), ~145 k lignes / 49 stations / ~10 ans.
- Cible **déséquilibrée** (~22 % de pluie) ⇒ piloter sur recall/F1 de la classe « pluie », pas l'accuracy.
- Prédicteurs clés cohérents physiquement : `Sunshine`, `Humidity3pm`, `Cloud3pm`, `Pressure`, plus station & saison.
- Les deux modèles atteignent ~0,85 d'accuracy / ~0,87–0,88 de ROC-AUC mais **ratent ~½ des jours de pluie** ;
  `class_weight="balanced"` remonte nettement le recall (au prix de l'accuracy) — arbitrage métier à trancher.

**Améliorations modèle.** gérer le déséquilibre (class weights / SMOTE / ajustement du seuil de décision),
feature engineering temporel (saison cyclique, retards), gestion fine de la forte missingness de `Sunshine`/`Cloud`
(indicateurs de manquant), validation **temporelle** (éviter de prédire le passé avec le futur), tuning (XGBoost/LightGBM).

**Vers le MLOps (roadmap projet).** ce notebook est le brouillon ; les étapes suivantes consistent à :
1. extraire ces traitements en scripts versionnés (`process.py`, `train.py`, `evaluate.py`) ;
2. tracer chaque entraînement avec **MLflow** (params, métriques, artefacts) ;
3. versionner données & modèles (**DVC**) ; 4. exposer une **API d'inférence** ; 5. conteneuriser (Docker Compose).